# Brainrot Transformer 
This is my implementation of Andrej Karpathy's "Building GPT from Scratch". All credits go to the original author. This notebook involves making the **Transformer** architecture from scratch but with my own twist. Instead of training it on the Tiny Shakespeare Dataset, this is trained on **Brainrot Corpus** which was generated by me using Claude and involves no web scraping. Both datatsets are similar in file size (1.1MB) and character count (1 million) for continuity purposes.

## Setting up and Inspecting the Dataset

In [1]:
# Opening and reading our dataset
with open('brainrot_corpus.txt','r') as f:
    text = f.read()

In [2]:
print("Length of dataset in terms of characters ",len(text))

Length of dataset in terms of characters  1115384


In [3]:
# Seeing the first couple characters
print(text[:1000])

CRASHOUT KAREN:
Bruh, it's giving rizz, Grimace Shake Guy
bro is cooked fr fr.
RIP my the AI slop, Sigma said six seven for
no reason with a rizz, don't leave me dry.

CHAT:
Rizzler, why you actin like a whole the
bazooka? don't leave me dry.

CHOPPELGANGER:
Yo, the skibidi is crazy, Chat lost all his
aura again, clock it - fr!
Deadass, aura minus 1000. Chat really yeeted
the bazooka in front of everybody.

CHOPPELGANGER:
Bro got rejected by the ohio energy and
pulled up with the the cap, rizz level 100.
Ngl, on god. Chat really farmed aura for no
reason in front of everybody.

MEWING MIKE:
Hold on, it's giving the doi doi doi, Mewing
Mike this is so ohio fr fr.
Sigma'S Little Brother is mewing in the
mirror and I said this is so ohio, that's
the the sauce.
Fam, did you hear Ohio Npc is goated with
the sauce? ohio final boss.

CROCODILO:
Chat said brainrot vibes and then farmed
aura for no reason.

GYATT GURL:
Sigma said aura at 0% and then is goated
with the sauce.
Hold on, Ohio Npc p

In [4]:
# to see all the unique characters that occur in this text
chars = list(set(text))
print(''.join(chars))
print(len(chars))

w LdK-P!Vsx:
nmlj1SHzOWD%hfEeI0q27pG6tvoR?BYTkN.,'gMaiFrZUXbu#cAyC
66


## Tokenization
The next step is to convert text to a sequence of integers. To keep things simple we are going to use character level tokenization meaning each character is represented by a different integer. The size of our vocabulary (ie number of unique tokens) will be the number of unique characters which is 66. Character tokenization has small vocabulary sizes but long sequences of tokens to represent a line as compared to sub-word encoding used by latest LLMs which convert a part of a word into a token (integer) and hence have a huge vocabulary but smaller sequences to represent the same string.

In [5]:
# making the vocabulary and the encode-decode functions
vocab_size = len(chars)
char_to_int = {c:i for i,c in enumerate(chars)}
int_to_char = {i:c for i,c in enumerate(chars)}
encode = lambda s: [char_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_char[i] for i in l)
eg = encode("hello")
print("encoding the sentence 'hello' ",eg)
print("decoding the same ",decode(eg))

encoding the sentence 'hello'  [25, 28, 15, 15, 39]
decoding the same  hello


In [6]:
# now encoding/tokenizing the entire dataset into a pytorch tensor
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([1115384]) torch.int64
tensor([65, 40, 63, 18, 19, 21, 57, 44,  1,  4, 63, 40, 27, 46, 11, 12, 42, 55,
        60, 25, 48,  1, 53, 37, 49,  9,  1, 50, 53, 38, 53, 13, 50,  1, 55, 53,
        20, 20, 48,  1, 35, 55, 53, 14, 52, 62, 28,  1, 18, 25, 52, 45, 28,  1,
        35, 60, 64, 12, 59, 55, 39,  1, 53,  9,  1, 62, 39, 39, 45, 28,  3,  1,
        26, 55,  1, 26, 55, 47, 12, 40, 29,  6,  1, 14, 64,  1, 37, 25, 28,  1,
        63, 29,  1,  9, 15, 39, 34, 48,  1, 18, 53, 50, 14, 52,  1,  9, 52, 53,
         3,  1,  9, 53, 10,  1,  9, 28, 38, 28, 13,  1, 26, 39, 55, 12, 13, 39,
         1, 55, 28, 52,  9, 39, 13,  1,  0, 53, 37, 25,  1, 52,  1, 55, 53, 20,
        20, 48,  1,  3, 39, 13, 49, 37,  1, 15, 28, 52, 38, 28,  1, 14, 28,  1,
         3, 55, 64, 47, 12, 12, 65, 19, 63, 44, 11, 12, 40, 53, 20, 20, 15, 28,
        55, 48,  1,  0, 25, 64,  1, 64, 39, 60,  1, 52, 62, 37, 53, 13,  1, 15,
        53, 45, 28,  1, 52,  1,  0, 25, 39, 15, 28,  1, 37, 25, 28, 12, 59, 52,
      

In [7]:
# finally splitting the data into training and validation, doing a 90% split
split = int(0.9*len(data))
train_data = data[:split]
val_data = data[split:]

## Batching
After tokenization comes the process of batching. While training the transformer we don't give it the entire dataset just once. Since it is learning to predict the next token (in this case the next character) we randomly take blocks of data of a fixed size and then provide the transformer multiple examples of the block to predict the next token (given only the first token of the block predict the second, given the first 2 predict the 3rd and so on...). A group of multiple blocks is called a batch.

In [11]:
# demonstration of how a single batch can be used to provide multiple training examples
block_size = 8
sample = data[:block_size]
target = data[1:block_size+1]
for i in range(block_size):
    context = sample[:i+1]
    response = target[i]
    print("When context is ",context," the target is ",response)

When context is  tensor([65])  the target is  tensor(40)
When context is  tensor([65, 40])  the target is  tensor(63)
When context is  tensor([65, 40, 63])  the target is  tensor(18)
When context is  tensor([65, 40, 63, 18])  the target is  tensor(19)
When context is  tensor([65, 40, 63, 18, 19])  the target is  tensor(21)
When context is  tensor([65, 40, 63, 18, 19, 21])  the target is  tensor(57)
When context is  tensor([65, 40, 63, 18, 19, 21, 57])  the target is  tensor(44)
When context is  tensor([65, 40, 63, 18, 19, 21, 57, 44])  the target is  tensor(1)


In [10]:
# when training a transformer we use multiple bactches called blocks
block_size = 8 # size of a sequence of data
batch_size = 4 # number of blocks
torch.manual_seed(1337)
def get_batch(split,batch_size=4,block_size=8):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:block_size+i] for i in ix])
    y = torch.stack([data[i+1:block_size+i+1] for i in ix])
    return x,y
x,y = get_batch('train')
print("size of x ",x.shape)
print(x)
print("size of y ",y.shape)
print(y)
# total number of training examples given by this entire batch = 4 * 8 = 32
for i in range(batch_size):
    for j in range(block_size):
        context = x[i][:j+1]
        target = y[i][j]
        print("When context is ",context," the target is ",target)

size of x  torch.Size([4, 8])
tensor([[37, 12, 52, 13,  3,  1, 37, 25],
        [39, 39, 45,  1, 37, 25, 28, 12],
        [ 1, 14, 39,  9, 31, 60, 53, 37],
        [ 1, 37, 25, 28,  1, 37, 25, 28]])
size of y  torch.Size([4, 8])
tensor([[12, 52, 13,  3,  1, 37, 25, 28],
        [39, 45,  1, 37, 25, 28, 12, 26],
        [14, 39,  9, 31, 60, 53, 37, 39],
        [37, 25, 28,  1, 37, 25, 28,  1]])
When context is  tensor([37])  the target is  tensor(12)
When context is  tensor([37, 12])  the target is  tensor(52)
When context is  tensor([37, 12, 52])  the target is  tensor(13)
When context is  tensor([37, 12, 52, 13])  the target is  tensor(3)
When context is  tensor([37, 12, 52, 13,  3])  the target is  tensor(1)
When context is  tensor([37, 12, 52, 13,  3,  1])  the target is  tensor(37)
When context is  tensor([37, 12, 52, 13,  3,  1, 37])  the target is  tensor(25)
When context is  tensor([37, 12, 52, 13,  3,  1, 37, 25])  the target is  tensor(28)
When context is  tensor([39])  the t

## Simple Langauge Modeling: Bigrams
Before diving into transformers, let's make the simplest language model possible: Bigrams, which are just fancy lookup tables.

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class Bigram(nn.Module): # subclass of nn.Module
    def __init__(self,vocab_size):
        super().__init__()
        # create a lookup table using which the last token of the input with predict logits for the next token
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
    def forward(self,idx,targets=None):
        # idx and targets are input,ground truth of size (B,T) where:
        # B: batch size (4), T: time ie length of a sequence (8)
        # predicting logits from the lookup table
        logits = self.token_embedding_table(idx) # (B,T,C) where C: channels ie no. of unique chars (66)
        B,T,C = logits.shape
        if targets is None:
            loss = None
        else:
            logits = logits.view(B*T,C) # changing dimensions to fit inside pytorch cross entropy function
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
        return logits,loss
    def generate(self,idx,max_possible_token):
        # the purpose of this function is to generate from the model
        for i in range(max_possible_token):
            logits,loss = self(idx) # get predictions
            # focus only on the last time step ie the last token
            logits = logits[:,-1,:] # (B,C)
            # get probabilites
            probs = F.softmax(logits,dim=1) # (B,C)
            # predict the most probable next token
            idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
            # concatenate to sequence
            idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
        return idx

In [13]:
m = Bigram(vocab_size)
logits,loss = m(x,y)
print(logits.shape)
print(loss) # loss is measured using cross entropy which is inverse log of number of chars so if logits are totally random:
# loss = -ln(1/66) = 4.189

torch.Size([32, 66])
tensor(4.5562, grad_fn=<NllLossBackward0>)


In [14]:
# let's see what so far we can generate
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),100)[0].tolist())) # we generate next 100 tokens on a tensor containing just 0
# unlike training where we have mutliple batches, when we are asking it to generate from a sequence we only have 1 batch so we extract
# it using [0] and convert it to a list of tokens which our decode function can handle.
# right now we expect the generated output to be nonsensical

wc--2VY:jyStg6cCVoRau2O6w-Sj1fxMMgi dBo0'cVpvahheKoiXOXwcYdyNbufRBCvVH
SWSG'umHdpgnIX2%qvlFSW0'oo%T#B


In [15]:
# now lets make a training function for bigram starting by initializing optimizer which adjusts weights based on gradients
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)
def train(rounds=10000):
    for i in range(rounds):
        xb,yb = get_batch('train',batch_size=32) # get batches from data
        optimizer.zero_grad(set_to_none=True)
        logits,loss = m.forward(xb,yb)
        loss.backward() # calculate all gradients
        optimizer.step() # adjust weights
    return loss.item()
loss = train()
print(loss)

2.427410840988159


In [16]:
# now that we have better loss lets generate something, it will not be coherent but look more gibberish language rather than random
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),100)[0].tolist()))

whe
sax? d fax gms, ar
d bix alluche
igre
COKar i thaiss MIMA whacrairis7#is

butewhe UTY:
Nhed isedo


## Self Attention Block
What makes Bigrams so inaccurate is that they make predictions by only looking at the last token while completely ignoring all the other tokens, and most importantly, the overall context. Self-attention is what makes transformers ingest context.

In [19]:
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape # example batch

torch.Size([4, 8, 2])

In self attention, at any given position, the token at that position only takes in the context from all the tokens that came before it in previous positions. It does not include future tokens in its context because during inference, future tokens are the ones to be predicted and hence don't exist at the time of prediction.

In [21]:
# let's try weakest form of making context: average
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # shape (t,C)
        xbow[b,t] = torch.mean(xprev,0)

The above average is very weak and inaccurate but also very inefficient as large datasets have a lot of batches and blocks so doing calculations one by one is extremely slow. Instead we use matrix multiplication as a weighted aggregation. Below is an example as to how that would look like.

In [23]:
a = torch.tril(torch.ones(3,3)) # torch.ones creates a 3x3 matrix where all elements are 1, tril only keeps the lower triangular matrix, rest
# everything becomes 0.
a = a / torch.sum(a,1,keepdim=True) # so that along dim 1 ie row, total sum of every element in each row must be 1
b = torch.randint(0,10,(3,2)).float() # creates a matrix of shape 3x2 where every element is chosen randomly between 0 to 10
c = a @ b # matrix multiplying a and b
print("a=")
print(a)
print("=======")
print("b=")
print(b)
print("=======")
print("a.b=")
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[0., 3.],
        [9., 9.],
        [8., 8.]])
a.b=
tensor([[0.0000, 3.0000],
        [4.5000, 6.0000],
        [5.6667, 6.6667]])


In [25]:
# instead of doing it like above however we do it with a mask
tril = torch.tril(torch.ones(T,T)) 
wei = torch.zeros(T,T) # weight matrix
wei = wei.masked_fill(tril==0,float('-inf')) # in a given position, if tril==0, then that position in wei should be -1
wei = F.softmax(wei,dim=-1) # softmax fixes it by changing all -inf to 0 and doing the same thing as a / torch.sum so that sum of all
# elements along each row is 1. We are doing the same thing above just like this since not all words in the context have the same weight
# some words matter more than others for the context of our current word, this is called affinity which is set in the wei matrix before 
# it undergoes this operation by the data, unlike being initialised by zeros like we have done in this example
xbow2 = wei @ x

Now we will actually implement a bare bones self attention head where instead of wei affinities being initialised to zero, we calculate them by taking dot product of query and key matrices and the final output is generated by matrix multiplying wei with the value matrix. Query, key and value matrices are generated by passing the tokens through a q,k,v linear dense layer respectively which is trained.

In [27]:
B,T,C = 4,8,32
x = torch.randn(B,T,C)
head_size=16 # dimension of vectors inside each attention head

query = nn.Linear(C,head_size,bias=False)
key = nn.Linear(C,head_size,bias=False)
value=nn.Linear(C,head_size,bias=False)

# All of them below are of shape (B,T,16)
q = query(x)
k = key(x)
v = value(x)

# Initialising wei affinities
wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,T,16).transpose --> (B,T,16) @ (B,16,T) --> (B,T,T)
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ v # (B,T,T) @ (B,T,16) --> (B,T,16)
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4593, 0.5407, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3200, 0.3219, 0.3581, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0268, 0.0210, 0.7122, 0.2400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1823, 0.2428, 0.1159, 0.0964, 0.3625, 0.0000, 0.0000, 0.0000],
        [0.2729, 0.0335, 0.0474, 0.1227, 0.5047, 0.0188, 0.0000, 0.0000],
        [0.1035, 0.2175, 0.0813, 0.1374, 0.0521, 0.3825, 0.0255, 0.0000],
        [0.1877, 0.0896, 0.0454, 0.3006, 0.1369, 0.0241, 0.0583, 0.1574]],
       grad_fn=<SelectBackward0>)

- Attention is a communication process where tokens exchange information based on weighted sums. However there is no communication between different batches as they are processed separately parallelly in different attention heads. 
- Self attention is when query, key and value come from the same source ie x, which is common in decoder only transformers, whereas in ecoder only transformers they come from different sources.
- Cross attention is different from self attention when we don't include mask, because in cross attention (used mostly in encoders) we're not predicting future tokens so during training we take future tokens in account in the context.
- In attention, tokens have no concept of position and space so we add positional embeddings as well.
- When calculating wei, we also divide it by root of head_size: ```wei = (q @ k.transpose(-2,-1)) * head_size**-0.5``` because when multiply q,k the values (affinities) are really large positives and really large negatives with high variance so softmax returns one value with weight 1 and others close to 0 which is really bad for training so dividing by root d makes variance 1 meaning softmax results are more distributed.